# PhishLens — Character-Level Deep Learning Model

A **character-level 1D-CNN** for phishing URL detection.  
Unlike the classical Random Forest baseline (which uses 9 hand-crafted lexical features),
this model operates directly on **raw URL text**, learning its own character-level
patterns — suspicious substrings, unusual character distributions, domain tricks —
end-to-end from the data.

**Architecture:**
```
URL chars → Embedding(32) → Conv1D(64, k=5) → GlobalMaxPool → Dense(64) → Dropout(0.3) → Dense(1, sigmoid)
```

**Dataset:** 11,430 URLs (50/50 phishing/legitimate), same as the baseline.

## 0. Setup & Imports

In [ ]:
import json
import string
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

# Paths
ML_DIR = Path(".").resolve()
DATA_DIR = ML_DIR / "data"
MODELS_DIR = ML_DIR / "models"
RAW_CSV = DATA_DIR / "raw_urls.csv"

# Hyper-parameters
MAX_URL_LEN = 200      # pad/truncate all URLs to this length
EMBEDDING_DIM = 32     # character embedding dimensionality
CONV_FILTERS = 64      # number of 1D conv filters
KERNEL_SIZE = 5        # conv kernel width
DENSE_UNITS = 64       # hidden dense layer size
DROPOUT_RATE = 0.3     # dropout for regularization
BATCH_SIZE = 64
EPOCHS = 20
RANDOM_STATE = 42

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1. Load Dataset

In [ ]:
df = pd.read_csv(RAW_CSV)
print(f"Dataset: {len(df):,} rows")
print(f"Label distribution:\n{df['label'].value_counts().to_string()}")
print(f"\nSample URLs:")
for _, row in df.sample(5, random_state=RANDOM_STATE).iterrows():
    tag = "PHISH" if row['label'] == 1 else "LEGIT"
    print(f"  [{tag}] {row['url'][:80]}")

## 2. Build Character Vocabulary

Scan every URL in the dataset and collect all unique printable ASCII characters.
Map each character to a unique integer index (1-indexed; 0 is reserved for padding).

In [ ]:
# Collect all unique characters across the dataset
all_chars = set()
for url in df["url"]:
    all_chars.update(url)

# Keep only printable ASCII characters, sorted for reproducibility
printable_set = set(string.printable)
vocab_chars = sorted(ch for ch in all_chars if ch in printable_set)

# Build char -> index mapping (1-indexed; 0 = padding)
char_to_idx = {ch: idx + 1 for idx, ch in enumerate(vocab_chars)}
vocab_size = len(char_to_idx) + 1  # +1 for the padding index 0

print(f"Vocabulary size: {vocab_size} (including padding token)")
print(f"Characters: {''.join(vocab_chars[:50])}{'...' if len(vocab_chars) > 50 else ''}")

# Save vocabulary for inference
MODELS_DIR.mkdir(parents=True, exist_ok=True)
vocab_path = MODELS_DIR / "char_vocab.json"
with open(vocab_path, "w") as f:
    json.dump({
        "char_to_idx": char_to_idx,
        "max_url_len": MAX_URL_LEN,
        "vocab_size": vocab_size,
    }, f, indent=2)
print(f"Saved vocabulary to: {vocab_path}")

## 3. Encode URLs as Integer Sequences

Each URL is converted to a fixed-length sequence of character indices:
- Characters not in the vocabulary are mapped to 0 (treated as padding)
- URLs shorter than `MAX_URL_LEN` are right-padded with 0s
- URLs longer than `MAX_URL_LEN` are truncated

In [ ]:
def encode_url(url: str, char_map: dict, max_len: int) -> np.ndarray:
    """Encode a single URL string into a fixed-length integer array."""
    encoded = [char_map.get(ch, 0) for ch in url[:max_len]]
    # Pad with 0s if shorter than max_len
    if len(encoded) < max_len:
        encoded += [0] * (max_len - len(encoded))
    return np.array(encoded, dtype=np.int32)


# Encode the full dataset
X = np.array([encode_url(url, char_to_idx, MAX_URL_LEN) for url in df["url"]])
y = df["label"].values.astype(np.float32)

print(f"X shape: {X.shape}  (samples × max_url_len)")
print(f"y shape: {y.shape}")
print(f"\nExample encoded URL (first 30 chars):")
print(f"  URL:     {df['url'].iloc[0][:30]}")
print(f"  Encoded: {X[0][:30]}")

## 4. Train/Test Split (80/20 Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {len(X_train):,} samples")
print(f"Test:  {len(X_test):,} samples")
print(f"Train label distribution: {np.bincount(y_train.astype(int))}")
print(f"Test  label distribution: {np.bincount(y_test.astype(int))}")

## 5. Build the Character-Level CNN Model

Architecture:
1. **Embedding** — maps each character index to a dense 32-dim vector
2. **Conv1D** — 64 filters, kernel size 5, detects local character n-gram patterns
3. **GlobalMaxPooling1D** — extracts the strongest activation per filter
4. **Dense(64, relu)** — learns non-linear feature combinations
5. **Dropout(0.3)** — regularization to prevent overfitting
6. **Dense(1, sigmoid)** — binary output (phishing probability)

In [ ]:
def build_char_cnn(vocab_size: int, max_len: int) -> keras.Model:
    """Build a character-level 1D-CNN for URL classification."""
    model = keras.Sequential([
        layers.Input(shape=(max_len,)),
        layers.Embedding(
            input_dim=vocab_size,
            output_dim=EMBEDDING_DIM,
            mask_zero=False,
            name="char_embedding",
        ),
        layers.Conv1D(
            filters=CONV_FILTERS,
            kernel_size=KERNEL_SIZE,
            activation="relu",
            padding="same",
            name="conv1d",
        ),
        layers.GlobalMaxPooling1D(name="global_max_pool"),
        layers.Dense(DENSE_UNITS, activation="relu", name="dense_hidden"),
        layers.Dropout(DROPOUT_RATE, name="dropout"),
        layers.Dense(1, activation="sigmoid", name="output"),
    ])
    
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_char_cnn(vocab_size, MAX_URL_LEN)
model.summary()

## 6. Train with Early Stopping

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    X_train, y_train,
    validation_split=0.15,    # 15% of training data for validation
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1,
)

## 7. Training History Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(history.history["loss"], label="Train Loss", linewidth=2)
ax1.plot(history.history["val_loss"], label="Val Loss", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Binary Crossentropy")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history.history["accuracy"], label="Train Acc", linewidth=2)
ax2.plot(history.history["val_accuracy"], label="Val Acc", linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Training & Validation Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_epoch = early_stopping.stopped_epoch - early_stopping.patience + 1 if early_stopping.stopped_epoch > 0 else len(history.history['loss'])
print(f"\nBest epoch: {best_epoch}")
print(f"Best val_loss: {min(history.history['val_loss']):.4f}")
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.4f}")

## 8. Evaluate on Test Set

In [ ]:
def false_positive_rate(y_true, y_pred):
    """FPR = FP / (FP + TN)"""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp / (fp + tn) if (fp + tn) > 0 else 0.0


# Get predictions
y_proba = model.predict(X_test, batch_size=BATCH_SIZE).flatten()
y_pred = (y_proba >= 0.5).astype(int)

# Compute metrics
deep_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-Score": f1_score(y_test, y_pred),
    "FPR": false_positive_rate(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba),
}

print("Deep Learning Model — Test Set Results")
print("=" * 45)
for metric, value in deep_metrics.items():
    print(f"  {metric:<12} {value:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"  TN={cm[0,0]:>5}  FP={cm[0,1]:>5}")
print(f"  FN={cm[1,0]:>5}  TP={cm[1,1]:>5}")

## 9. Comparison: Deep Learning vs. Classical Baseline

Load the baseline Random Forest metrics from `baseline_metadata.json` and print
a side-by-side comparison table.

In [ ]:
# Load baseline metrics
baseline_meta_path = MODELS_DIR / "baseline_metadata.json"
with open(baseline_meta_path) as f:
    baseline_meta = json.load(f)

baseline_metrics = baseline_meta["metrics"]

# Print comparison table
print("=" * 80)
print("  MODEL COMPARISON — Classical Baseline vs. Deep Learning")
print("  Dataset: 11,430 URLs (50/50 phishing/legit) | Split: 80/20 stratified")
print("=" * 80)
print()
print(f"  {'Approach':<28} {'Features':<30}")
print(f"  {'Random Forest (Baseline)':<28} {'9 hand-crafted lexical':<30}")
print(f"  {'Char-CNN (Deep Learning)':<28} {'Raw URL characters (200 max)':<30}")
print()

header = f"  {'Metric':<14} {'RF Baseline':>12} {'Char-CNN':>12} {'Δ':>10}"
print(header)
print("  " + "-" * 50)

for metric in ["Accuracy", "Precision", "Recall", "F1-Score", "FPR", "ROC-AUC"]:
    bl = baseline_metrics.get(metric, 0)
    dl = deep_metrics.get(metric, 0)
    delta = dl - bl
    # For FPR, lower is better, so flip the indicator
    if metric == "FPR":
        indicator = "✓" if delta < 0 else "✗" if delta > 0 else "="
    else:
        indicator = "✓" if delta > 0 else "✗" if delta < 0 else "="
    print(f"  {metric:<14} {bl:>12.4f} {dl:>12.4f} {delta:>+9.4f} {indicator}")

print("  " + "-" * 50)
print()

# Highlight winner per F1
bl_f1 = baseline_metrics.get("F1-Score", 0)
dl_f1 = deep_metrics.get("F1-Score", 0)
if dl_f1 > bl_f1:
    print(f"  ★ Deep Learning model WINS by F1-Score: {dl_f1:.4f} vs {bl_f1:.4f} (+{dl_f1-bl_f1:.4f})")
elif dl_f1 < bl_f1:
    print(f"  ★ Baseline model still leads by F1-Score: {bl_f1:.4f} vs {dl_f1:.4f}")
    print(f"    (Deep model is a complementary technique, not a replacement)")
else:
    print(f"  ★ Models are tied on F1-Score: {dl_f1:.4f}")

print()
print("  Key insight: These models use ENTIRELY DIFFERENT features.")
print("  The baseline uses hand-crafted lexical statistics.")
print("  The deep model learns directly from raw character sequences.")
print("  Ensembling them can capture complementary patterns.")
print("=" * 80)

## 10. Save Model & Metadata

In [ ]:
# Save the trained model
model_path = MODELS_DIR / "deep_model.h5"
model.save(model_path)
print(f"Saved model to: {model_path}")

# Save deep model metadata
deep_metadata = {
    "model_name": "Character-Level CNN",
    "model_type": "deep-learning-char-cnn",
    "description": (
        "Character-level 1D-CNN trained directly on raw URL text. "
        "No hand-crafted features — the model learns character-level patterns "
        "(suspicious substrings, domain tricks, etc.) end-to-end from data."
    ),
    "architecture": {
        "input": f"URL encoded as {MAX_URL_LEN}-length integer sequence",
        "layers": [
            f"Embedding(vocab_size={vocab_size}, dim={EMBEDDING_DIM})",
            f"Conv1D(filters={CONV_FILTERS}, kernel_size={KERNEL_SIZE}, relu)",
            "GlobalMaxPooling1D",
            f"Dense({DENSE_UNITS}, relu)",
            f"Dropout({DROPOUT_RATE})",
            "Dense(1, sigmoid)",
        ],
    },
    "training": {
        "optimizer": "adam",
        "loss": "binary_crossentropy",
        "epochs_trained": len(history.history["loss"]),
        "max_epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "early_stopping_patience": 3,
        "validation_split": 0.15,
    },
    "vocab": {
        "vocab_size": vocab_size,
        "max_url_len": MAX_URL_LEN,
        "file": "char_vocab.json",
    },
    "label_encoding": {"legitimate": 0, "phishing": 1},
    "dataset_size": len(df),
    "train_size": len(X_train),
    "test_size": len(X_test),
    "metrics": {k: round(v, 4) for k, v in deep_metrics.items()},
    "baseline_comparison": {
        "baseline_model": baseline_meta["model_name"],
        "baseline_metrics": {k: round(v, 4) if isinstance(v, float) else v
                            for k, v in baseline_metrics.items()},
    },
    "files": {
        "model": "deep_model.h5",
        "vocabulary": "char_vocab.json",
        "metadata": "deep_metadata.json",
    },
}

deep_meta_path = MODELS_DIR / "deep_metadata.json"
with open(deep_meta_path, "w") as f:
    json.dump(deep_metadata, f, indent=2)
print(f"Saved metadata to: {deep_meta_path}")

## 11. Quick Inference Test

Run the saved model on a few sample URLs to verify it works end-to-end.

In [ ]:
# Reload model and vocab to simulate fresh inference
loaded_model = keras.models.load_model(model_path)

with open(vocab_path) as f:
    loaded_vocab = json.load(f)
loaded_char_map = loaded_vocab["char_to_idx"]
loaded_max_len = loaded_vocab["max_url_len"]

test_urls = [
    ("https://www.google.com", "should be legit"),
    ("http://mybank-login-secure.verify-account.com", "should be phishing"),
    ("https://xn--pple-43d.com", "punycode, should be phishing"),
    ("https://github.com/tensorflow/tensorflow", "should be legit"),
    ("http://secure-paypal-login.com/update?id=12345", "should be phishing"),
]

print("Quick Inference Test — Character-Level CNN")
print("=" * 85)
for url, expected in test_urls:
    encoded = encode_url(url, loaded_char_map, loaded_max_len)
    encoded_batch = np.expand_dims(encoded, axis=0)  # add batch dim
    prob = loaded_model.predict(encoded_batch, verbose=0)[0, 0]
    verdict = "PHISHING" if prob >= 0.5 else "LEGIT"
    print(f"  {url:<55} → {verdict:<10} (conf: {max(prob, 1-prob):.2%})  [{expected}]")
print("=" * 85)

## 12. Summary

| Aspect | Baseline (Random Forest) | Deep Model (Char-CNN) |
|--------|--------------------------|----------------------|
| **Input** | 9 hand-crafted lexical features | Raw URL characters (200 max) |
| **Technique** | Classical ML (tree ensemble) | Deep Learning (1D-CNN) |
| **Feature Engineering** | Manual (entropy, dot count, etc.) | Learned end-to-end |
| **Strengths** | Fast, interpretable, works on known patterns | Can discover novel character-level patterns |
| **Weaknesses** | Limited to pre-defined features | Needs more data to shine, less interpretable |
| **Complementary?** | ✓ Yes — different feature spaces mean ensembling captures more signal |

Both models are now saved and can be loaded by the backend inference engine.